In [ ]:
# Breast Cancer scRNA-seq: Data Processing Pipeline

#Full preprocessing pipeline for GSE161529 — downloading, loading, merging, 
#QC filtering, normalization, dimensionality reduction, clustering, and cell 
#type annotation for 13 normal and 8 TNBC tumor samples.

#**Note:** Raw data must be downloaded from GEO before running this notebook.
#Dataset: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE161529  
#Total runtime: ~45-60 minutes depending on connection speed.

In [1]:
import os
print(os.path.exists("../raw_data/GSE161529_RAW.tar"))
print(os.path.exists("../data/breast_cancer_annotated.h5ad"))

False
True


In [2]:
import scanpy as sc
import os

# Create data directories if they don't exist
os.makedirs('../raw_data', exist_ok=True)
os.makedirs('../data', exist_ok=True)

print("Directories ready")
print("Scanpy version:", sc.__version__)

Directories ready
Scanpy version: 1.11.5


/var/folders/m0/my66jx5974n5jwyr68bndpg80000gn/T/ipykernel_82465/2035273673.py:9: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("Scanpy version:", sc.__version__)


In [ ]:
import urllib.request
import os

url = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE161nnn/GSE161529/suppl/GSE161529_RAW.tar"
output_path = "../raw_data/GSE161529_RAW.tar"

print("Downloading dataset... this may take several minutes")
urllib.request.urlretrieve(url, output_path)
print(f"Download complete! File size: {os.path.getsize(output_path) / 1e9:.2f} GB")

In [ ]:
import tarfile
import os

if len([f for f in os.listdir("../raw_data/") if f.endswith('.mtx.gz')]) > 0:
    print("Files already extracted, skipping")
    files = os.listdir("../raw_data/")
else:
    print("Extracting files...")
    with tarfile.open("../raw_data/GSE161529_RAW.tar", "r") as tar:
        tar.extractall("../raw_data/")
    files = os.listdir("../raw_data/")
    print(f"Extraction complete! Found {len(files)} files")

In [ ]:
import os

files = os.listdir("../raw_data/")
print(f"Total files: {len(files)}")
for f in sorted(files)[:20]:
    print(f)

In [ ]:
# Separate tumor and normal samples
normal = [f for f in files if '_N-' in f and 'Total' in f]
tumor = [f for f in files if '_T-' in f and 'Total' in f]

print(f"Normal samples: {len(normal)}")
print(f"Tumor samples: {len(tumor)}")
print("\nFirst few normal files:")
for f in sorted(normal)[:6]:
    print(f)
print("\nFirst few tumor files:")
for f in sorted(tumor)[:6]:
    print(f)

In [ ]:
# Look at all unique prefixes to find tumor samples
print("All unique sample identifiers:")
for f in sorted(files):
    if 'matrix' in f:
        print(f)
        

In [ ]:
normal = [f for f in files if f.startswith('GSM') and '_N-' in f and 'Total' in f and 'matrix' in f]
tumor_tn = [f for f in files if '_TN-' in f and 'matrix' in f]

print(f"Normal samples: {len(normal)}")
print(f"Triple Negative tumor samples: {len(tumor_tn)}")

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os

raw_data_path = "../raw_data/"

def load_sample(sample_name, condition):
    """Load a single 10x sample from barcodes + matrix files"""
    # Get the base name without matrix.mtx.gz
    base = sample_name.replace('-matrix.mtx.gz', '')
    
    matrix_file = os.path.join(raw_data_path, f"{base}-matrix.mtx.gz")
    barcodes_file = os.path.join(raw_data_path, f"{base}-barcodes.tsv.gz")
    
    # Read matrix
    adata = sc.read_mtx(matrix_file).T
    
    # Read barcodes
    barcodes = pd.read_csv(barcodes_file, header=None)[0].values
    adata.obs_names = barcodes
    
    # Add metadata
    adata.obs['sample'] = base
    adata.obs['condition'] = condition
    
    return adata

print("Functions defined, ready to load samples")

In [ ]:
# Load all normal samples
print("Loading normal samples...")
normal_adatas = []
for f in sorted(normal):
    print(f"  Loading {f}...")
    adata = load_sample(f, 'Normal')
    normal_adatas.append(adata)

print(f"\nLoaded {len(normal_adatas)} normal samples")

# Load all TN tumor samples
print("\nLoading tumor samples...")
tumor_adatas = []
for f in sorted(tumor_tn):
    print(f"  Loading {f}...")
    adata = load_sample(f, 'Tumor_TN')
    tumor_adatas.append(adata)

print(f"\nLoaded {len(tumor_adatas)} tumor samples")

In [ ]:
# Merge all samples together
print("Merging all samples...")
all_adatas = normal_adatas + tumor_adatas
adata = all_adatas[0].concatenate(all_adatas[1:], batch_key='sample_id')

print(adata)
print(f"\nCondition counts:")
print(adata.obs['condition'].value_counts())

In [ ]:
print("Gene names:", adata.var_names[:10].tolist())
print("Total genes:", adata.n_vars)

In [ ]:
features_path = "../raw_data/GSE161529_features.tsv.gz"

if os.path.exists(features_path):
    print("Features file already exists, skipping download")
else:
    print("Downloading features file...")
    urllib.request.urlretrieve(features_url, features_path)
    print("Done!")

In [ ]:
import pandas as pd

# Load features file
features = pd.read_csv("../raw_data/GSE161529_features.tsv.gz", 
                       header=None, sep='\t')
print("Features file shape:", features.shape)
print(features.head())

In [ ]:
# Add gene names to adata
adata.var_names = features[1].values
adata.var['gene_ids'] = features[0].values

print("Gene names added!")
print("First 5 genes:", adata.var_names[:5].tolist())
print(adata)

In [ ]:
adata.write('../data/breast_cancer_raw.h5ad')
print("Saved!")

In [ ]:
# Breast Cancer scRNA-seq: Data Processing Pipeline

In [ ]:
import matplotlib.pyplot as plt

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=80, facecolor='white')

# Basic filtering
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

print(adata)

In [ ]:
adata.var_names_make_unique()
print("Variable names made unique")
print(adata)


In [ ]:
# Calculate mitochondrial gene percentage
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

# Plot QC metrics
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True, groupby='condition')

In [ ]:
print("Before filtering:", adata.n_obs, "cells")

adata = adata[adata.obs.n_genes_by_counts < 6000, :]
adata = adata[adata.obs.n_genes_by_counts > 200, :]
adata = adata[adata.obs.total_counts < 40000, :]
adata = adata[adata.obs.pct_counts_mt < 25, :]

print("After filtering:", adata.n_obs, "cells")
print("\nCondition counts after filtering:")
print(adata.obs['condition'].value_counts())

In [ ]:
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True, groupby='condition')

In [ ]:
adata.write('../data/breast_cancer_qc.h5ad')
print("QC complete and saved!")
print(f"Final dataset: {adata.n_obs} cells x {adata.n_vars} genes")

In [ ]:
## Quality Control

In [ ]:
# Normalization
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Highly variable genes
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5, batch_key='sample_id')
sc.pl.highly_variable_genes(adata)

print(f"Highly variable genes: {adata.var.highly_variable.sum()}")

In [ ]:
## Normalization and Feature Selection

In [ ]:
adata.raw = adata
adata = adata[:, adata.var.highly_variable]

sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt'])
sc.pp.scale(adata, max_value=10)

sc.tl.pca(adata, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata, log=True)

print(adata)

In [ ]:
adata.write('../data/breast_cancer_pca.h5ad')
print("Saved!")

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
sc.tl.umap(adata)
sc.pl.umap(adata, color='condition', title='Tumor vs Normal')

In [ ]:
sc.tl.leiden(adata, resolution=0.5)
sc.pl.umap(adata, color=['leiden', 'condition'], ncols=2)

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon')
sc.pl.rank_genes_groups(adata, n_genes=5, sharey=False)

In [ ]:
## Dimensionality Reduction and Clustering

In [ ]:
cell_type_map = {
    '0': 'Immune cells',
    '1': 'Luminal epithelial',
    '2': 'Basal epithelial',
    '3': 'Luminal epithelial 2',
    '4': 'Stressed cells',
    '5': 'Stromal cells',
    '6': 'Macrophages',
    '7': 'Monocytes',
    '8': 'Proliferating cells',
    '9': 'High MT cells',
    '10': 'Ribosomal high',
    '11': 'Endothelial/Stromal',
    '12': 'Fibroblasts',
    '13': 'Epithelial cells',
    '14': 'Plasma B cells',
    '15': 'Dendritic cells',
    '16': 'Mast cells',
    '17': 'CAFs',
    '18': 'NK/B cells',
    '19': 'Endothelial cells',
    '20': 'T/NK cells'
}

adata.obs['cell_type'] = adata.obs['leiden'].map(cell_type_map)
sc.pl.umap(adata, color='cell_type', legend_loc='on data', 
           title='Breast Cancer Cell Types', legend_fontsize=8)


In [ ]:
adata.write('../data/breast_cancer_annotated.h5ad')

# Plot cell types alongside condition
sc.pl.umap(adata, color=['cell_type', 'condition'], 
           ncols=2, legend_fontsize=7,
           save='_breast_cancer_celltypes.png')

In [ ]:
import pandas as pd

# Calculate cell type proportions by condition
proportions = adata.obs.groupby(['cell_type', 'condition']).size().unstack(fill_value=0)
proportions['total'] = proportions.sum(axis=1)
proportions['pct_tumor'] = (proportions['Tumor_TN'] / proportions['total'] * 100).round(1)
proportions['pct_normal'] = (proportions['Normal'] / proportions['total'] * 100).round(1)

print(proportions[['Normal', 'Tumor_TN', 'pct_normal', 'pct_tumor']].sort_values('pct_tumor', ascending=False))

In [ ]:
import matplotlib.pyplot as plt

# Plot proportion bar chart
ax = proportions[['Normal', 'Tumor_TN']].sort_values('Tumor_TN').plot(
    kind='barh', 
    stacked=True,
    figsize=(10, 8),
    color=['#4C72B0', '#DD8452']
)
plt.title('Cell Type Composition: Normal vs Tumor_TN', fontsize=14)
plt.xlabel('Number of cells')
plt.tight_layout()
plt.savefig('../figures/cell_type_proportions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved!")

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=80, facecolor='white')

adata = sc.read('../data/breast_cancer_annotated.h5ad')
print(adata)

In [ ]:
marker_genes = {
    'Luminal epithelial': ['KRT18', 'KRT7'],
    'Basal epithelial': ['KRT5', 'KRT17'],
    'Fibroblasts': ['DCN', 'LUM'],
    'CAFs': ['COL1A1', 'COL1A2'],
    'Macrophages': ['LYZ', 'CD74'],
    'Proliferating cells': ['HMGB2', 'STMN1'],
    'Stressed cells': ['DDIT4', 'HMOX1'],
    'Endothelial cells': ['VWF', 'PECAM1'],
    'Plasma B cells': ['MZB1', 'JCHAIN'],
    'T/NK cells': ['NKG7', 'GZMB']
}

sc.pl.dotplot(adata, marker_genes, groupby='cell_type', 
              dendrogram=False,
              title='Marker Gene Expression by Cell Type',
              save='_marker_genes.png')